# Traveller Planetary System Generator

This Jupyter Notebook implements a Python-based application to procedurally generate detailed planetary systems according to the rules set forth in the *Traveller World Builder's Handbook*.

The development follows the comprehensive workplan provided, structured in a phased approach:
1. **Foundational Architecture**: Defining core data structures and utility functions.
2. **Stellar Body Generation**: Creating the system's star or stars.
3. **System Architecture & Population**: Determining the number of worlds and their possible orbital locations.
4. **World Placement & Finalization**: Placing the worlds into orbits and calculating orbital parameters.
5. **Planetary Body Detailing**: Sizing worlds and generating their satellite systems.
6. **System Output & Presentation**: Formatting the generated data into readable profiles and tables.

The final output is a comprehensive, human-readable table generated using the Pandas library, mirroring the style of the IISS survey forms in the handbook.

## I. Foundational Architecture: Data Structures & Core Utilities

### 1.0 Imports and Setup

This cell imports all necessary libraries for the project.

In [ ]:
import random
import math
import dataclasses
from dataclasses import dataclass, field
from typing import List, Dict, Any, Tuple, Optional
import pandas as pd
import numpy as np
from IPython.display import SVG, display

# Configure pandas for better display in the notebook
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

### 1.1 Core Utility Library

This section contains low-level functions for dice rolling and essential astronomical conversions (Orbit# to AU and vice-versa), as described in the workplan.

In [ ]:
class Utils:
    """Houses all common, low-level functions."""
    
    @staticmethod
    def D6(n=1): # 1D, 2D, etc.
        return sum(random.randint(1, 6) for _ in range(n))

    @staticmethod
    def D3():
        return random.randint(1, 3)
        
    @staticmethod
    def d10(): # Returns 0-9
        return random.randint(0, 9)

    @staticmethod
    def eHex(value):
        if 0 <= value <= 9:
            return str(value)
        ehex_map = {
            10: 'A', 11: 'B', 12: 'C', 13: 'D', 14: 'E', 15: 'F', 16: 'G',
            17: 'H', 18: 'J', 19: 'K', 20: 'L', 21: 'M', 22: 'N', 23: 'P',
            24: 'Q', 25: 'R', 26: 'S', 27: 'T', 28: 'U', 29: 'V', 30: 'W',
            31: 'X', 32: 'Y', 33: 'Z'
        }
        return ehex_map.get(value, str(value))

    ORBIT_TABLE = {
        0: {'dist': 0.0, 'diff': 0.4},
        1: {'dist': 0.4, 'diff': 0.3},
        2: {'dist': 0.7, 'diff': 0.3},
        3: {'dist': 1.0, 'diff': 0.6},
        4: {'dist': 1.6, 'diff': 1.2},
        5: {'dist': 2.8, 'diff': 2.4},
        6: {'dist': 5.2, 'diff': 4.8},
        7: {'dist': 10.0, 'diff': 10.0},
        8: {'dist': 20.0, 'diff': 20.0},
        9: {'dist': 40.0, 'diff': 37.0},
        10: {'dist': 77.0, 'diff': 77.0},
        11: {'dist': 154.0, 'diff': 154.0},
        12: {'dist': 308.0, 'diff': 307.0},
        13: {'dist': 615.0, 'diff': 615.0},
        14: {'dist': 1230.0, 'diff': 1270.0},
        15: {'dist': 2500.0, 'diff': 2400.0},
        16: {'dist': 4900.0, 'diff': 4900.0},
        17: {'dist': 9800.0, 'diff': 9700.0},
        18: {'dist': 19500.0, 'diff': 20000.0},
        19: {'dist': 39500.0, 'diff': 39200.0},
        20: {'dist': 78700.0, 'diff': 0.0} # End of table
    }

    @classmethod
    def orbit_to_au(cls, orbit_num: float) -> float:
        """Converts Traveller Orbit# to Astronomical Units (AU)."""
        if orbit_num < 0: return 0.0
        whole_orbit = math.floor(orbit_num)
        fractional_part = orbit_num - whole_orbit
        
        base_dist = cls.ORBIT_TABLE.get(whole_orbit, {}).get('dist', 0)
        diff = cls.ORBIT_TABLE.get(whole_orbit, {}).get('diff', 0)
        
        return base_dist + (diff * fractional_part)

    @classmethod
    def au_to_orbit(cls, au: float) -> float:
        """Converts AU to Traveller Orbit#."""
        if au <= 0: return 0.0
        
        full_orbit = 0
        for i in range(21):
            if au >= cls.ORBIT_TABLE[i]['dist']:
                full_orbit = i
            else:
                break

        base_dist = cls.ORBIT_TABLE[full_orbit]['dist']
        diff = cls.ORBIT_TABLE[full_orbit]['diff']
        if diff == 0: return float(full_orbit)

        fractional_part = (au - base_dist) / diff
        return full_orbit + fractional_part

    @staticmethod
    def calculate_hzco(luminosity: float) -> float:
        """Calculates the Habitable Zone Center Orbit# (HZCO) in AU."""
        hzco_au = math.sqrt(luminosity)
        return Utils.au_to_orbit(hzco_au)

    @staticmethod
    def from_eHex(ehex_char):
        if ehex_char.isdigit():
            return int(ehex_char)
        ehex_map = {
            'A': 10, 'B': 11, 'C': 12, 'D': 13, 'E': 14, 'F': 15, 'G': 16,
            'H': 17, 'J': 18, 'K': 19, 'L': 20, 'M': 21, 'N': 22, 'P': 23,
            'Q': 24, 'R': 25, 'S': 26, 'T': 27, 'U': 28, 'V': 29, 'W': 30,
            'X': 31, 'Y': 32, 'Z': 33
        }
        return ehex_map.get(ehex_char.upper(), 0)

### 1.2 System Data Model Definition

This cell defines the object-oriented data model using Python's `@dataclass`. 

In [ ]:
@dataclass
class Satellite:
    designation: str = ""
    parent_body: Any = None # Forward reference
    is_ring: bool = False
    size_code: str = ""
    orbit_pd: float = 0.0
    period_hours: float = 0.0
    notes: List[str] = field(default_factory=list)
    diameter_km: float = 0.0
    mass_terran: float = 0.0
    gravity: float = 0.0
    atmosphere_code: str = ""
    hydrographics_code: str = ""
    provisional_temp: str = ""

@dataclass
class UWP:
    starport: str = "X"
    size: str = "0"
    atmosphere: str = "0"
    hydrographics: str = "0"
    population: str = "0"
    government: str = "0"
    law_level: str = "0"
    tech_level: str = "0"

    def __str__(self):
        return f"{self.starport}{self.size}{self.atmosphere}{self.hydrographics}{self.population}{self.government}{self.law_level}-{self.tech_level}"

@dataclass
class CulturalProfile:
    diversity: int = 0
    xenophilia: int = 0
    uniqueness: int = 0
    symbology: int = 0
    cohesion: int = 0
    progressiveness: int = 0
    expansionism: int = 0
    militancy: int = 0

@dataclass
class Sophont:
    name: str = ""
    homeworld_hex: str = ""
    physical_characteristics: List[str] = field(default_factory=list)
    cultural_profile: CulturalProfile = field(default_factory=CulturalProfile)

@dataclass
class Polity:
    name: str = ""
    capital_hex: str = ""
    controlled_systems: List[str] = field(default_factory=list)

@dataclass
class Wave:
    name: str = ""
    origin_hex: str = ""
    age_centuries: int = 0
    wave_type: str = "" # "thin" or "thick"
    propagation_rate: float = 0.0

@dataclass
class PlanetaryBody:
    name: str = ""
    designation: str = ""
    parent_star_group: str = "" 
    body_type: str = "" 
    orbit_num: float = 0.0
    orbit_au: float = 0.0
    eccentricity: float = 0.0
    inclination: float = 0.0
    period_years: float = 0.0
    size_code: str = ""
    diameter_km: float = 0.0
    mass_terran: float = 0.0
    mean_temperature: float = 0.0
    surface_features: str = ""
    life_details: str = ""
    satellites: List[Satellite] = field(default_factory=list)
    notes: List[str] = field(default_factory=list)
    atmosphere_code: str = ""
    hydrographics_code: str = ""
    provisional_temp: str = ""
    uwp: UWP = field(default_factory=UWP)
    cultural_profile: CulturalProfile = field(default_factory=CulturalProfile)
    is_mainworld: bool = False

@dataclass
class Star:
    designation: str = ""
    is_composite: bool = False
    components: List[str] = field(default_factory=list)
    spectral_type: str = ""
    mass: float = 0.0
    diameter: float = 0.0
    luminosity: float = 0.0
    temp_k: float = 0.0
    parent: Optional['Star'] = None
    orbit_class: str = "" 
    orbit_num: float = 0.0
    eccentricity: float = 0.0
    period_years: float = 0.0
    hzco: float = 0.0
    mao: float = 0.0
    available_orbits: List[Tuple[float, float]] = field(default_factory=list)
    orbiting_bodies: List[PlanetaryBody] = field(default_factory=list)

@dataclass
class StellarSystem:
    name: str = "Generated System"
    age_gyr: float = 0.0
    stars: List[Star] = field(default_factory=list)
    gas_giant_count: int = 0
    planetoid_belt_count: int = 0
    terrestrial_planet_count: int = 0
    empty_orbit_count: int = 0
    anomalous_planets: list = field(default_factory=list)
    total_worlds: int = 0
    baseline_number: int = 0
    baseline_orbit: float = 0.0
    spread: float = 0.0

    @property
    def primary_star(self):
        return self.stars[0] if self.stars else None
    
    @property
    def all_worlds(self):
        worlds = []
        for star in self.stars:
            if not star.is_composite:
                worlds.extend(star.orbiting_bodies)
        return worlds

@dataclass
class Sector:
    name: str = "Generated Sector"
    width: int = 8
    height: int = 10
    systems: Dict[str, StellarSystem] = field(default_factory=dict)
    native_sophonts: Dict[str, Sophont] = field(default_factory=dict)
    settlement_waves: List[Wave] = field(default_factory=list)

### 1.3 Data Tables from Handbook

This cell centralizes all data tables from the *World Builder's Handbook* into Python data structures.

In [ ]:
DATA = {
    'star_type_determination': {
        'Type': {2: 'Special', 3: 'M', 4: 'M', 5: 'M', 6: 'M', 7: 'K', 8: 'K', 9: 'G', 10: 'G', 11: 'F', 12: 'Hot'},
        'Hot': {2: 'A', 3: 'A', 4: 'A', 5: 'A', 6: 'A', 7: 'A', 8: 'A', 9: 'A', 10: 'B', 11: 'B', 12: 'O'},
        'Special': {2: 'A', 3: 'Class VI', 4: 'Class VI', 5: 'Class VI', 6: 'Class IV', 7: 'Class IV', 8: 'Class IV', 9: 'Class III', 10: 'Class III', 11: 'Giants', 12: 'Giants'},
        'Unusual': {2: 'Peculiar', 3: 'Peculiar', 4: 'Class IV', 5: 'BD', 6: 'BD', 7: 'BD', 8: 'D', 9: 'D', 10: 'D', 11: 'Class III', 12: 'Giants'},
        'Giants': {2: 'Class III', 3: 'Class III', 4: 'Class III', 5: 'Class III', 6: 'Class III', 7: 'Class III', 8: 'Class II', 9: 'Class II', 10: 'Class II', 11: 'Class Ib', 12: 'Class Ia'},
        'Peculiar': {1: 'Black Hole', 2: 'Neutron Star', 3: 'Pulsar', 4: 'Protostar', 5: 'Nebula', 6: 'Star Cluster'}
    },
    'star_subtype': {
        'Numeric': {2: 0, 3: 1, 4: 3, 5: 5, 6: 7, 7: 9, 8: 8, 9: 6, 10: 4, 11: 2, 12: 0},
        'M-type': {2: 8, 3: 6, 4: 5, 5: 4, 6: 0, 7: 2, 8: 1, 9: 3, 10: 5, 11: 7, 12: 9}
    },
    'star_mass': {
        'Ia': {'O0': 200, 'O5': 80, 'B0': 60, 'B5': 30, 'A0': 20, 'A5': 15, 'F0': 13, 'F5': 12, 'G0': 12, 'G5': 13, 'K0': 14, 'K5': 18, 'M0': 20, 'M5': 25, 'M9': 30},
        'Ib': {'O0': 150, 'O5': 60, 'B0': 40, 'B5': 25, 'A0': 15, 'A5': 13, 'F0': 12, 'F5': 10, 'G0': 10, 'G5': 11, 'K0': 12, 'K5': 13, 'M0': 15, 'M5': 20, 'M9': 25},
        'II': {'O0': 130, 'O5': 40, 'B0': 30, 'B5': 20, 'A0': 14, 'A5': 11, 'F0': 10, 'F5': 8, 'G0': 8, 'G5': 10, 'K0': 10, 'K5': 12, 'M0': 14, 'M5': 16, 'M9': 18},
        'III': {'O0': 110, 'O5': 30, 'B0': 20, 'B5': 10, 'A0': 8, 'A5': 6, 'F0': 4, 'F5': 3, 'G0': 2.5, 'G5': 2.4, 'K0': 1.1, 'K5': 1.5, 'M0': 1.8, 'M5': 2.4, 'M9': 8},
        'IV': {'B0': 20, 'B5': 10, 'A0': 4, 'A5': 2.3, 'F0': 2, 'F5': 1.5, 'G0': 1.7, 'G5': 1.2, 'K0': 1.5},
        'V': {'O0': 90, 'O5': 60, 'B0': 18, 'B5': 5, 'A0': 2.2, 'A5': 1.8, 'F0': 1.5, 'F5': 1.3, 'G0': 1.1, 'G5': 0.9, 'K0': 0.8, 'K5': 0.7, 'M0': 0.5, 'M5': 0.16, 'M9': 0.08},
        'VI': {'O0': 2, 'O5': 1.5, 'B0': 0.5, 'B5': 0.4, 'G0': 0.8, 'G5': 0.7, 'K0': 0.6, 'K5': 0.5, 'M0': 0.4, 'M5': 0.12, 'M9': 0.075}
    },
    'star_temp': {
        'Ia': {'O0': 50000, 'O5': 40000, 'B0': 25000, 'B5': 14000, 'A0': 9500, 'A5': 8500, 'F0': 7500, 'F5': 6500, 'G0': 5500, 'G5': 4700, 'K0': 4000, 'K5': 3500, 'M0': 3200, 'M5': 3000, 'M9': 2800},
        'Ib': {'O0': 50000, 'O5': 40000, 'B0': 22000, 'B5': 13000, 'A0': 9200, 'A5': 8200, 'F0': 7200, 'F5': 6200, 'G0': 5300, 'G5': 4500, 'K0': 3800, 'K5': 3300, 'M0': 3000, 'M5': 2800, 'M9': 2600},
        'II': {'O0': 50000, 'O5': 38000, 'B0': 20000, 'B5': 12000, 'A0': 9000, 'A5': 8000, 'F0': 7000, 'F5': 6000, 'G0': 5100, 'G5': 4300, 'K0': 3600, 'K5': 3100, 'M0': 2800, 'M5': 2600, 'M9': 2400},
        'III': {'O0': 48000, 'O5': 36000, 'B0': 18000, 'B5': 11000, 'A0': 8500, 'A5': 7500, 'F0': 6500, 'F5': 5500, 'G0': 4800, 'G5': 4100, 'K0': 3400, 'K5': 3000, 'M0': 2600, 'M5': 2400, 'M9': 2200},
        'IV': {'B0': 28000, 'B5': 14000, 'A0': 9500, 'A5': 7800, 'F0': 7000, 'F5': 6300, 'G0': 5800, 'G5': 5500, 'K0': 5000},
        'V': {'O0': 50000, 'O5': 40000, 'B0': 30000, 'B5': 15000, 'A0': 10000, 'A5': 8000, 'F0': 7500, 'F5': 6500, 'G0': 6000, 'G5': 5600, 'K0': 5200, 'K5': 4400, 'M0': 3700, 'M5': 3000, 'M9': 2400},
        'VI': {'O0': 40000, 'O5': 30000, 'B0': 20000, 'B5': 12000, 'G0': 5800, 'G5': 5400, 'K0': 5000, 'K5': 4200, 'M0': 3500, 'M5': 2800, 'M9': 2200}
    },
    'non_primary_star_determination': {
        2: {'Secondary': 'Other', 'Companion': 'Other', 'Post-Stellar': 'D*'},
        3: {'Secondary': 'Other', 'Companion': 'Other', 'Post-Stellar': 'D'},
        4: {'Secondary': 'Random', 'Companion': 'Random', 'Post-Stellar': 'D'},
        5: {'Secondary': 'Random', 'Companion': 'Random', 'Post-Stellar': 'D'},
        6: {'Secondary': 'Random', 'Companion': 'Lesser', 'Post-Stellar': 'D'},
        7: {'Secondary': 'Lesser', 'Companion': 'Lesser', 'Post-Stellar': 'D'},
        8: {'Secondary': 'Lesser', 'Companion': 'Sibling', 'Post-Stellar': 'BD'},
        9: {'Secondary': 'Sibling', 'Companion': 'Sibling', 'Post-Stellar': 'BD'},
        10: {'Secondary': 'Sibling', 'Companion': 'Twin', 'Post-Stellar': 'BD'},
        11: {'Secondary': 'Twin', 'Companion': 'Twin', 'Post-Stellar': 'BD'},
        12: {'Secondary': 'Twin', 'Companion': 'Twin', 'Post-Stellar': 'BD'}
    },
    'star_diameter': {
        'Ia': {'O0': 25, 'O5': 22, 'B0': 20, 'B5': 60, 'A0': 120, 'A5': 180, 'F0': 210, 'F5': 280, 'G0': 330, 'G5': 360, 'K0': 420, 'K5': 600, 'M0': 900, 'M5': 1200, 'M9': 1800},
        'Ib': {'O0': 24, 'O5': 20, 'B0': 14, 'B5': 25, 'A0': 50, 'A5': 75, 'F0': 85, 'F5': 115, 'G0': 135, 'G5': 150, 'K0': 180, 'K5': 260, 'M0': 380, 'M5': 600, 'M9': 800},
        'II': {'O0': 22, 'O5': 18, 'B0': 12, 'B5': 14, 'A0': 30, 'A5': 45, 'F0': 50, 'F5': 66, 'G0': 77, 'G5': 90, 'K0': 110, 'K5': 160, 'M0': 230, 'M5': 350, 'M9': 500},
        'III': {'O0': 21, 'O5': 15, 'B0': 10, 'B5': 6, 'A0': 5, 'A5': 5, 'F0': 5, 'F5': 5, 'G0': 10, 'G5': 15, 'K0': 20, 'K5': 40, 'M0': 60, 'M5': 100, 'M9': 200},
        'IV': {'B0': 8, 'B5': 5, 'A0': 4, 'A5': 3, 'F0': 3, 'F5': 2, 'G0': 3, 'G5': 4, 'K0': 6},
        'V': {'O0': 20, 'O5': 12, 'B0': 7, 'B5': 3.5, 'A0': 2.2, 'A5': 2.0, 'F0': 1.7, 'F5': 1.5, 'G0': 1.1, 'G5': 0.95, 'K0': 0.9, 'K5': 0.8, 'M0': 0.7, 'M5': 0.2, 'M9': 0.1},
        'VI': {'O0': 0.18, 'O5': 0.18, 'B0': 0.2, 'B5': 0.5, 'G0': 0.8, 'G5': 0.7, 'K0': 0.6, 'K5': 0.5, 'M0': 0.4, 'M5': 0.1, 'M9': 0.08}
    },
    'gas_giant_quantity': {
        'roll_map': {4: 1, 5: 2, 6: 2, 7: 3, 8: 3, 9: 4, 10: 4, 11: 4, 12: 5, 13: 6},
        'min_roll': 4, 'max_roll': 13
    },
    'planetoid_belt_quantity': {
        'roll_map': {6: 1, 7: 2, 8: 2, 9: 2, 10: 2, 11: 2, 12: 3},
        'min_roll': 6, 'max_roll': 12
    },
    'eccentricity_values': {
        5: {'base': -0.001, 'roll': lambda: Utils.D6() / 1000},
        7: {'base': 0.00, 'roll': lambda: Utils.D6() / 200},
        9: {'base': 0.03, 'roll': lambda: Utils.D6() / 100},
        10: {'base': 0.05, 'roll': lambda: Utils.D6(2) / 20},
        11: {'base': 0.05, 'roll': lambda: Utils.D6(2) / 20},
        12: {'base': 0.30, 'roll': lambda: Utils.D6(2) / 20},
    },
    'terrestrial_sizing': {
        1: lambda: Utils.D6(),
        2: lambda: Utils.D6(),
        3: lambda: Utils.D6(2),
        4: lambda: Utils.D6(2),
        5: lambda: Utils.D6(2) + 3,
        6: lambda: Utils.D6(2) + 3
    },
    'terrestrial_world_sizing': {
        '1D_roll': {1: '1D', 2: '1D', 3: '2D', 4: '2D', 5: '2D+3', 6: '2D+3'},
        'size_ranges': {
            '1D': {1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6},
            '2D': {2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12},
            '2D+3': {5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14, 15: 15}
        }
    },
    'gas_giant_sizing': {
        'GS': {'d_roll': lambda: Utils.D3() + Utils.D3(), 'm_roll': lambda: 5 * (Utils.D6() + 1)},
        'GM': {'d_roll': lambda: Utils.D6() + 6, 'm_roll': lambda: 20 * (Utils.D6(3) - 1)},
        'GL': {'d_roll': lambda: Utils.D6(2) + 6, 'm_roll': lambda: Utils.D3() * 50 * (Utils.D6(3) + 4)}
    },
    'significant_moon_quantity': {
        'planet_size_1_2': lambda: Utils.D6() - 5,
        'planet_size_3_9': lambda: Utils.D6(2) - 8,
        'planet_size_a_f': lambda: Utils.D6(2) - 6,
        'small_gas_giant': lambda: Utils.D6(3) - 7,
        'medium_large_gas_giant': lambda: Utils.D6(4) - 6,
        'dms': {
            'orbit_less_than_1': -1,
            'adjacent_companion': -1,
            'adjacent_unavailability': -1,
            'adjacent_outermost': -1
        }
    }
}

## II. Phase 1: Stellar Body Generation

In [ ]:
def generate_world_name():
    prefixes = ["Ard", "Bor", "Cor", "Den", "Eth", "Fen", "Gor", "Hen", "Ish", "Jen", "Kel", "Lor", "Mor", "Nor", "Orr", "Per", "Quor", "Ren", "Sor", "Tor", "Ur", "Ver", "Wor", "Xen", "Yor", "Zor"]
    suffixes = ["ia", "os", "a", "us", "is", "en", "or", "an", "el", "ar"]
    return random.choice(prefixes) + random.choice(suffixes)

def _interpolate_stellar_data(spectral_str, class_v_data):
    s_type = spectral_str[0]
    s_subtype = int(spectral_str[1])
    types = ['O', 'B', 'A', 'F', 'G', 'K', 'M']
    subtypes = [0, 5, 9] if s_type == 'M' else [0, 5]
    lower_subtype = max([s for s in subtypes if s <= s_subtype])
    upper_subtype_list = [s for s in subtypes if s > s_subtype]
    upper_subtype = min(upper_subtype_list) if upper_subtype_list else -1
    if upper_subtype == -1: 
        current_type_index = types.index(s_type)
        if current_type_index + 1 >= len(types): return class_v_data[f"{s_type}{lower_subtype}"]
        next_type = types[current_type_index + 1]
        lower_key = f"{s_type}{lower_subtype}"
        upper_key = f"{next_type}0"
        span = 10 - lower_subtype
        pos = s_subtype - lower_subtype
    else:
        lower_key = f"{s_type}{lower_subtype}"
        upper_key = f"{s_type}{upper_subtype}"
        span = upper_subtype - lower_subtype
        pos = s_subtype - lower_subtype
    interp_ratio = pos / span if span > 0 else 0
    lower_val = class_v_data[lower_key]
    upper_val = class_v_data[upper_key]
    return lower_val + (upper_val - lower_val) * interp_ratio

def generate_primary_star(special_roll=None) -> Star:
    primary = Star(designation="A")
    roll = special_roll if special_roll else Utils.D6(2)
    star_type_result = DATA['star_type_determination']['Type'].get(min(roll, 12))
    if star_type_result == 'Hot':
        hot_roll = Utils.D6(2)
        star_type_result = DATA['star_type_determination']['Hot'].get(hot_roll)
        primary.spectral_type = "V"
    elif star_type_result == 'Special':
        star_type_result = 'G' 
        primary.spectral_type = "V"
    else:
        primary.spectral_type = "V"
    subtype_roll = Utils.D6(2)
    if star_type_result == 'M':
        subtype = DATA['star_subtype']['M-type'][subtype_roll]
    else:
        subtype = DATA['star_subtype']['Numeric'][subtype_roll]
    spectral_str = f"{star_type_result}{subtype}"
    primary.spectral_type = f"{spectral_str} {primary.spectral_type}"
    lum_class = primary.spectral_type.split(' ')[1]
    primary.mass = _interpolate_stellar_data(spectral_str, DATA['star_mass'][lum_class])
    primary.temp_k = _interpolate_stellar_data(spectral_str, DATA['star_temp'][lum_class])
    primary.diameter = _interpolate_stellar_data(spectral_str, DATA['star_diameter'][lum_class])
    temp_ratio = primary.temp_k / 5772
    primary.luminosity = (primary.diameter ** 2) * (temp_ratio ** 4)
    primary.hzco = Utils.calculate_hzco(primary.luminosity)
    primary.mao = Utils.au_to_orbit(0.01 * primary.diameter) 
    return primary

def _determine_non_primary_star_type(parent_star: Star, orbit_class: str) -> dict:
    dm = 0
    if parent_star.spectral_type.split(' ')[1] in ['III', 'IV']: dm -=1
    roll = Utils.D6(2) + dm
    category = 'Companion' if orbit_class == 'Companion' else 'Secondary'
    result = DATA['non_primary_star_determination'][min(12, max(2,roll))][category]
    return {'type': result, 'roll': roll}

def generate_stellar_system_stars(system: StellarSystem):
    primary = generate_primary_star()
    system.stars.append(primary)
    star_presense_dm = 0
    if primary.spectral_type[0] in ['O', 'B', 'A', 'F']: star_presense_dm += 1
    if primary.spectral_type[0] == 'M': star_presense_dm -= 1
    if Utils.D6(2) + star_presense_dm >= 10:
        close_star = Star(designation="B", parent=primary, orbit_class="Close")
        system.stars.append(close_star)
    if Utils.D6(2) + star_presense_dm >= 10:
        near_star = Star(designation="C", parent=primary, orbit_class="Near")
        system.stars.append(near_star)
    if Utils.D6(2) + star_presense_dm >= 10:
        far_star = Star(designation="D", parent=primary, orbit_class="Far")
        system.stars.append(far_star)
    for star in system.stars[:]: 
        companion_dm = 0
        if star.spectral_type and star.spectral_type[0] in ['O', 'B', 'A', 'F']: companion_dm += 1
        if star.spectral_type and star.spectral_type[0] == 'M': companion_dm -= 1
        if Utils.D6(2) + companion_dm >= 10:
            companion = Star(designation=f"{star.designation}b", parent=star, orbit_class="Companion")
            star.designation = f"{star.designation}a"
            system.stars.append(companion)
    for star in system.stars:
        if star.mass == 0.0 and star.parent: 
            result = _determine_non_primary_star_type(star.parent, star.orbit_class)
            star_type_info = result['type']
            if star_type_info == 'Random':
                new_star = generate_primary_star(special_roll=result['roll'])
                if new_star.mass > star.parent.mass: star_type_info = 'Lesser' 
                else:
                    star.spectral_type = new_star.spectral_type
                    star.mass = new_star.mass
                    star.diameter = new_star.diameter
                    star.luminosity = new_star.luminosity
                    star.temp_k = new_star.temp_k
            if star_type_info == 'Lesser':
                parent_type = star.parent.spectral_type[0]
                types = ['O', 'B', 'A', 'F', 'G', 'K', 'M']
                current_index = types.index(parent_type)
                if current_index + 1 < len(types):
                    new_type = types[current_index+1]
                    subtype = Utils.d10()
                    star.spectral_type = f"{new_type}{subtype} V"
                else:
                    star.spectral_type = f"M{Utils.d10()} V"
            elif star_type_info == 'Sibling':
                parent_type = star.parent.spectral_type.split(' ')[0]
                parent_subtype = int(parent_type[1:])
                new_subtype = parent_subtype - Utils.D6()
                new_type = parent_type[0]
                if new_subtype < 0:
                    types = ['O', 'B', 'A', 'F', 'G', 'K', 'M']
                    current_index = types.index(new_type)
                    if current_index + 1 < len(types):
                        new_type = types[current_index + 1]
                        new_subtype += 10
                    else:
                        new_subtype = 0 
                star.spectral_type = f"{new_type}{new_subtype} V"
            elif star_type_info == 'Twin':
                star.spectral_type = star.parent.spectral_type
                star.mass = star.parent.mass * (1 - (Utils.D6()-1)/100)
                star.diameter = star.parent.diameter * (1 - (Utils.D6()-1)/100)
            if star.mass == 0.0:
                if not star.spectral_type: star.spectral_type = "M0 V"
                lum_class = star.spectral_type.split(' ')[1]
                spectral_str = star.spectral_type.split(' ')[0]
                star.mass = _interpolate_stellar_data(spectral_str, DATA['star_mass'][lum_class])
                star.temp_k = _interpolate_stellar_data(spectral_str, DATA['star_temp'][lum_class])
                star.diameter = _interpolate_stellar_data(spectral_str, DATA['star_diameter'][lum_class])
            if star.luminosity == 0.0:
                temp_ratio = star.temp_k / 5772
                star.luminosity = (star.diameter ** 2) * (temp_ratio ** 4)
    for star in system.stars[:]:
        if star.orbit_class == "Companion" and star.parent:
            parent = star.parent
            composite_designation = parent.designation[:-1] + 'ab'
            if not any(s.designation == composite_designation for s in system.stars):
                composite = Star(
                    designation=composite_designation,
                    is_composite=True,
                    components=[parent.designation, star.designation],
                    mass=parent.mass + star.mass,
                    luminosity=parent.luminosity + star.luminosity,
                    spectral_type=parent.spectral_type 
                )
                composite.hzco = Utils.calculate_hzco(composite.luminosity)
                composite.mao = 0.5 + star.eccentricity 
                system.stars.append(composite)
    lifespan = 10 / (system.primary_star.mass ** 2.5) if system.primary_star.mass > 0 else 10
    if lifespan > 13.8: system.age_gyr = Utils.D6() * 2 + Utils.D3() - 1
    else: system.age_gyr = lifespan * (Utils.d10() / 10.0)
    system.age_gyr = round(max(0.1, min(system.age_gyr, 13.5)), 3)

## III. Phase 2: System Architecture and Population

In [ ]:
def determine_world_counts(system: StellarSystem):
    is_class_v_star = system.primary_star.spectral_type.endswith(' V')
    is_brown_dwarf = system.primary_star.spectral_type.startswith('BD')
    is_post_stellar = system.primary_star.spectral_type.startswith('D')
    is_protostar = system.primary_star.spectral_type.startswith(('T', 'P'))
    if Utils.D6(2) <= 9: 
        dm_quantity = 0
        if is_class_v_star and len(system.stars) == 1: dm_quantity += 1 
        if is_brown_dwarf: dm_quantity -= 2
        if is_post_stellar: dm_quantity -= 2
        if len(system.stars) >= 4: dm_quantity -= 1 
        roll = Utils.D6(2) + dm_quantity
        roll = max(DATA['gas_giant_quantity']['min_roll'], min(roll, DATA['gas_giant_quantity']['max_roll']))
        system.gas_giant_count = DATA['gas_giant_quantity']['roll_map'][roll]
    if Utils.D6(2) >= 8: 
        dm_quantity = 0
        if system.gas_giant_count > 0: dm_quantity += 1
        if is_protostar: dm_quantity += 3
        if is_post_stellar: dm_quantity += 1
        if len(system.stars) >= 2: dm_quantity += 1 
        roll = Utils.D6(2) + dm_quantity
        roll = max(DATA['planetoid_belt_quantity']['min_roll'], min(roll, DATA['planetoid_belt_quantity']['max_roll']))
        system.planetoid_belt_count = DATA['planetoid_belt_quantity']['roll_map'][roll]
    dm_quantity = 0
    if is_post_stellar: dm_quantity -= 1 
    roll = Utils.D6(2) - 2 + dm_quantity
    if roll < 3: system.terrestrial_planet_count = Utils.D3() + 2
    else: system.terrestrial_planet_count = roll + Utils.D3() - 1
    system.total_worlds = system.gas_giant_count + system.planetoid_belt_count + system.terrestrial_planet_count

def _calculate_hill_sphere_orbits(system: StellarSystem) -> List[Tuple[float, float]]:
    hill_spheres = {}
    for star in system.stars:
        if star.parent: 
            au_distance = Utils.orbit_to_au(star.orbit_num)
            hill_radius_au = au_distance * (1 - star.eccentricity) * (star.mass / (3 * star.parent.mass))**(1/3)
            hill_spheres[star.designation] = hill_radius_au
        else: 
            closest_secondary_au = float('inf')
            for other_star in system.stars:
                if other_star.parent == star: closest_secondary_au = min(closest_secondary_au, Utils.orbit_to_au(other_star.orbit_num))
            if closest_secondary_au != float('inf'):
                hill_radius_au = closest_secondary_au * (1 - star.eccentricity) * (star.mass / (3 * system.stars[0].mass))**(1/3)
                hill_spheres[star.designation] = hill_radius_au
            else: hill_spheres[star.designation] = float('inf') 
    stability_spheres_orbit = {s: Utils.au_to_orbit(hs / 3) for s, hs in hill_spheres.items()}
    forbidden_zones = []
    for star in system.stars:
        if star.designation in stability_spheres_orbit:
            orbit_val = stability_spheres_orbit[star.designation]
            forbidden_zones.append((star.orbit_num - orbit_val, star.orbit_num + orbit_val))
    forbidden_zones.sort()
    merged_zones = []
    if forbidden_zones:
        current_start, current_end = forbidden_zones[0]
        for next_start, next_end in forbidden_zones[1:]:
            if next_start <= current_end: current_end = max(current_end, next_end)
            else: merged_zones.append((current_start, current_end)); current_start, current_end = next_start, next_end
        merged_zones.append((current_start, current_end))
    available_orbits = []
    current_orbit = system.primary_star.mao 
    for zone_start, zone_end in merged_zones:
        if current_orbit < zone_start: available_orbits.append((current_orbit, zone_start))
        current_orbit = max(current_orbit, zone_end)
    if current_orbit < 20.0: available_orbits.append((current_orbit, 20.0))
    return available_orbits

def calculate_available_orbits(system: StellarSystem, model='simple'):
    primary_group = next((s for s in system.stars if not s.parent), None)
    if not primary_group: return
    if model == 'simple':
        forbidden_zones = []
        secondaries = [s for s in system.stars if s.parent and s.orbit_class != 'Companion']
        for star in secondaries:
            exclusion_start, exclusion_end = star.orbit_num - 1.0, star.orbit_num + 1.0
            if star.eccentricity > 0.2: exclusion_start -= 1.0; exclusion_end += 1.0
            if star.eccentricity > 0.5 and star.orbit_class in ['Close', 'Near']: exclusion_start -= 1.0; exclusion_end += 1.0
            forbidden_zones.append((exclusion_start, exclusion_end))
        forbidden_zones.sort()
        merged_zones = []
        if forbidden_zones:
            current_start, current_end = forbidden_zones[0]
            for next_start, next_end in forbidden_zones[1:]:
                if next_start <= current_end: current_end = max(current_end, next_end)
                else: merged_zones.append((current_start, current_end)); current_start, current_end = next_start, next_end
            merged_zones.append((current_start, current_end))
        available = []
        current_orbit = primary_group.mao
        for zone_start, zone_end in merged_zones:
            if current_orbit < zone_start: available.append((current_orbit, zone_start))
            current_orbit = max(current_orbit, zone_end)
        if current_orbit < 20.0: available.append((current_orbit, 20.0))
        primary_group.available_orbits = available
    elif model == 'physics': primary_group.available_orbits = _calculate_hill_sphere_orbits(system)

def calculate_baseline_and_spread(system: StellarSystem):
    primary_group = next((s for s in system.stars if not s.parent), None)
    if not primary_group: return
    dm = 0
    if any(s.orbit_class == 'Companion' for s in system.stars): dm -= 2
    if primary_group.spectral_type.endswith((' Ia', ' Ib', ' II')): dm += 3
    elif primary_group.spectral_type.endswith(' III'): dm += 2
    elif primary_group.spectral_type.endswith(' IV'): dm += 1
    elif primary_group.spectral_type.endswith(' VI'): dm -= 1
    if primary_group.spectral_type.startswith('D'): dm -= 2
    if system.total_worlds < 6: dm -= 4
    elif system.total_worlds <= 9: dm -= 3
    elif system.total_worlds <= 12: dm -= 2
    elif system.total_worlds <= 15: dm -= 1
    elif 18 <= system.total_worlds <= 20: dm += 1
    elif system.total_worlds > 20: dm += 2
    for star in system.stars: 
        if star != primary_group and not star.is_composite: dm -= 1
    system.baseline_number = Utils.D6(2) + dm
    if 1 <= system.baseline_number <= system.total_worlds: system.baseline_orbit = primary_group.hzco + (Utils.D6(2) - 7) / 10.0
    elif system.baseline_number < 1: system.baseline_orbit = primary_group.hzco - system.baseline_number + (Utils.D6(2) - 2) / 10.0
    else: system.baseline_orbit = primary_group.hzco - (system.baseline_number - system.total_worlds) + (Utils.D6(2) - 7) / 5.0
    is_available = any(start <= system.baseline_orbit <= end for start, end in primary_group.available_orbits)
    if not is_available:
        closest_dist, new_orbit, variance_roll = float('inf'), system.baseline_orbit, (Utils.D6(2) - 7) / 10.0
        for start, end in primary_group.available_orbits:
            if system.baseline_orbit < start: 
                if start - system.baseline_orbit < closest_dist: closest_dist = start - system.baseline_orbit; new_orbit = start + variance_roll
            elif system.baseline_orbit > end: 
                if system.baseline_orbit - end < closest_dist: closest_dist = system.baseline_orbit - end; new_orbit = end + variance_roll
            else: new_orbit = system.baseline_orbit; break
        system.baseline_orbit = new_orbit
    baseline_num_for_calc = max(1, system.baseline_number)
    numerator = system.baseline_orbit - primary_group.mao
    system.spread = numerator / baseline_num_for_calc if numerator > 0 and baseline_num_for_calc > 0 else 0.5

def handle_anomalies_and_empties(system: StellarSystem):
    empty_roll = Utils.D6(2)
    if empty_roll == 10: system.empty_orbit_count = 1
    elif empty_roll == 11: system.empty_orbit_count = 2
    elif empty_roll == 12: system.empty_orbit_count = 3
    else: system.empty_orbit_count = 0
    anomalous_roll = Utils.D6(2)
    num_anomalous = 0
    if anomalous_roll == 10: num_anomalous = 1
    elif anomalous_roll == 11: num_anomalous = 2
    elif anomalous_roll == 12: num_anomalous = 3
    for _ in range(num_anomalous):
        anomaly_type_roll = Utils.D6(2)
        anomaly_type = 'random'
        if anomaly_type_roll == 8: anomaly_type = 'eccentric'
        elif anomaly_type_roll == 9: anomaly_type = 'inclined'
        elif 10 <= anomaly_type_roll <= 11: anomaly_type = 'retrograde'
        elif anomaly_type_roll == 12: anomaly_type = 'trojan'
        system.anomalous_planets.append({'type': anomaly_type})
        system.terrestrial_planet_count += 1; system.total_worlds += 1

## IV. Phase 3: World Placement and Orbital Finalization

In [ ]:
def generate_orbital_slots(system: StellarSystem) -> List[dict]:
    primary_group = next((s for s in system.stars if not s.parent), None)
    if not primary_group: return []
    total_slots_needed = system.total_worlds + system.empty_orbit_count
    slots, current_orbit = [], primary_group.mao
    for i in range(total_slots_needed - len(system.anomalous_planets)):
        if i + 1 == system.baseline_number: current_orbit = system.baseline_orbit
        else: current_orbit += system.spread
        for start, end in primary_group.available_orbits:
            if current_orbit > end and start > (current_orbit - system.spread): 
                current_orbit = start + (current_orbit - end); break
        slots.append({'orbit_num': round(current_orbit, 2), 'type': 'regular'})
    for anomaly in system.anomalous_planets:
        if primary_group.available_orbits:
            zone = random.choice(primary_group.available_orbits)
            ano_orbit = random.uniform(zone[0], zone[1])
            slots.append({'orbit_num': round(ano_orbit, 2), 'type': 'anomalous', 'anomaly': anomaly})
    return sorted(slots, key=lambda x: x['orbit_num'])

def place_worlds(system: StellarSystem, orbital_slots: List[dict]):
    slots_map = {i: {'slot': s, 'body': None} for i, s in enumerate(orbital_slots)}
    placements = [{'type': 'Empty', 'count': system.empty_orbit_count}, {'type': 'Gas Giant', 'count': system.gas_giant_count}, {'type': 'Planetoid Belt', 'count': system.planetoid_belt_count}]
    num_slots = len(slots_map)
    available_slots_indices = list(range(num_slots))
    random.shuffle(available_slots_indices)
    for item in placements:
        count_to_place, placed_count = item['count'], 0
        while placed_count < count_to_place and available_slots_indices:
            slot_idx = available_slots_indices.pop(0)
            if slots_map[slot_idx]['body'] is not None: available_slots_indices.append(slot_idx); continue
            if slots_map[slot_idx]['slot']['type'] == 'anomalous' and item['type'] == 'Empty': available_slots_indices.append(slot_idx); continue
            slots_map[slot_idx]['body'] = item['type']
            placed_count += 1
    for slot_idx in available_slots_indices: slots_map[slot_idx]['body'] = 'Terrestrial'
    primary_group = next((s for s in system.stars if not s.parent), None)
    for i in range(num_slots):
        slot_info = slots_map[i]
        if slot_info['body'] == 'Empty': continue
        body = PlanetaryBody(name=generate_world_name(), parent_star_group=primary_group.designation, body_type=slot_info['body'], orbit_num=slot_info['slot']['orbit_num'])
        if slot_info['slot'].get('anomaly'): body.notes.append(f"Anomalous Orbit: {slot_info['slot']['anomaly']['type']}")
        body.orbit_au = Utils.orbit_to_au(body.orbit_num)
        if body.body_type != 'Planetoid Belt':
            ecc_roll = Utils.D6(2)
            if slot_info['slot'].get('anomaly'):
                anomaly_type = slot_info['slot']['anomaly']['type']
                if anomaly_type == 'eccentric': ecc_roll += 4
                elif anomaly_type == 'retrograde': body.notes.append("Retrograde Orbit")
                elif anomaly_type == 'inclined': body.inclination = (Utils.D6(2) * 5) + Utils.D6(); body.notes.append(f"Inclined Orbit: {body.inclination} degrees")
                elif anomaly_type == 'trojan': body.notes.append("Trojan Orbit")
            ecc_data = DATA['eccentricity_values'].get(min(12, max(5, ecc_roll)))
            if ecc_data: body.eccentricity = round(max(0.0, min(0.999, ecc_data['base'] + ecc_data['roll']())), 3)
        total_mass_for_period = sum(s.mass for s in system.stars if s.designation in primary_group.components) if primary_group.is_composite else primary_group.mass
        if body.body_type == 'Gas Giant': total_mass_for_period += 0.005
        body.period_years = math.sqrt(body.orbit_au**3 / total_mass_for_period)
        primary_group.orbiting_bodies.append(body)

## V. Phase 4: Planetary Body Detailing

In [ ]:
def detail_placed_worlds(system: StellarSystem):
    for world in system.all_worlds:
        if world.body_type == 'Terrestrial':
            roll_1d = Utils.D6()
            roll_type = DATA['terrestrial_world_sizing']['1D_roll'][roll_1d]
            size_roll = DATA['terrestrial_world_sizing']['size_ranges'][roll_type][Utils.D6() if roll_type == '1D' else (Utils.D6(2) if roll_type == '2D' else Utils.D6(2) + 3)]
            world.size_code = Utils.eHex(size_roll)
        elif world.body_type == 'Gas Giant':
            category_roll = Utils.D6()
            category = 'GS' if category_roll <= 2 else ('GM' if category_roll <= 4 else 'GL')
            d_roll = DATA['gas_giant_sizing'][category]['d_roll']()
            world.diameter_km = d_roll * 12800; world.mass_terran = DATA['gas_giant_sizing'][category]['m_roll'](); world.size_code = f"{category}{Utils.eHex(d_roll)}"
        elif world.body_type == 'Planetoid Belt': world.size_code = '0'
        num_moons, dm_moons = 0, 0
        if world.orbit_num < 1.0: dm_moons += DATA['significant_moon_quantity']['dms']['orbit_less_than_1']
        if world.body_type == 'Terrestrial':
            size_val = int(world.size_code, 16) if world.size_code in 'ABCDEF' else int(world.size_code)
            num_moons = max(0, (DATA['significant_moon_quantity']['planet_size_1_2']() if size_val <= 2 else (DATA['significant_moon_quantity']['planet_size_3_9']() if size_val <= 9 else DATA['significant_moon_quantity']['planet_size_a_f']())) + dm_moons)
        elif 'G' in world.size_code:
            num_moons = max(0, (DATA['significant_moon_quantity']['small_gas_giant']() if 'GS' in world.size_code else DATA['significant_moon_quantity']['medium_large_gas_giant']()) + dm_moons)
        for i in range(num_moons):
            sat = Satellite(parent_body=world)
            size_roll = Utils.D6()
            if size_roll <= 3: sat.size_code = 'S'
            elif size_roll <= 5: r_roll = Utils.D3() - 1; sat.size_code = 'R' if r_roll == 0 else str(r_roll); sat.is_ring = (r_roll == 0)
            else:
                if world.body_type == 'Terrestrial': size_val = int(world.size_code, 16) if world.size_code in 'ABCDEF' else int(world.size_code); sat.size_code = Utils.eHex(max(0, size_val - 1 - Utils.D6()))
                elif 'G' in world.size_code: sat.size_code = Utils.eHex(Utils.D6())
            world.satellites.append(sat)

def flag_points_of_interest(system: StellarSystem):
    primary_group = next((s for s in system.stars if not s.parent), None)
    if not primary_group: return
    hz_min, hz_max = primary_group.hzco - 1.0, primary_group.hzco + 1.0
    for world in system.all_worlds:
        if world.notes and 'Anomalous' in world.notes[0]: world.notes.append("Point of Interest: Anomalous orbit.")
        for satellite in world.satellites:
            is_large_moon = False
            try: 
                if satellite.size_code.isdigit() and int(satellite.size_code) >= 4: is_large_moon = True
            except ValueError: pass
            if is_large_moon and hz_min <= world.orbit_num <= hz_max: 
                satellite.notes.append("Point of Interest: Large moon in Habitable Zone. Mainworld Candidate."); world.notes.append(f"Candidate moon {satellite.designation}")

def _to_roman_numeral(num: int) -> str:
    if num <= 0: return str(num)
    roman_map = {1000: 'M', 900: 'CM', 500: 'D', 400: 'CD', 100: 'C', 90: 'XC', 50: 'L', 40: 'XL', 10: 'X', 9: 'IX', 5: 'V', 4: 'IV', 1: 'I'}
    roman_numeral = ""
    for value, numeral in roman_map.items():
        while num >= value: roman_numeral += numeral; num -= value
    return roman_numeral

def assign_final_designations(system: StellarSystem):
    for star_group in system.stars:
        if star_group.is_composite: continue
        planet_counter, belt_counter = 1, 1
        star_group.orbiting_bodies.sort(key=lambda x: x.orbit_num)
        for world in star_group.orbiting_bodies:
            if world.body_type == 'Planetoid Belt': world.designation = f"{star_group.designation} P{_to_roman_numeral(belt_counter)}"; belt_counter += 1
            elif world.body_type in ['Terrestrial', 'Gas Giant']: world.designation = f"{star_group.designation} {_to_roman_numeral(planet_counter)}"; planet_counter += 1
            moon_char_code = ord('a')
            for satellite in world.satellites:
                satellite.designation = f"{world.designation} {chr(moon_char_code)}"; moon_char_code += 1

## VI. Phase 5: System Output and Presentation

In [ ]:
def generate_short_profile(system: StellarSystem) -> str:
    return f"{system.gas_giant_count}-{system.planetoid_belt_count}-{system.terrestrial_planet_count}-{system.baseline_number}-{round(system.spread, 1)}"

def generate_long_profile(system: StellarSystem) -> str:
    profile_parts = []
    for star_group in system.stars:
        if star_group.is_composite: continue
        star_profile, world_types = f"{star_group.designation}-", []
        for world in star_group.orbiting_bodies: world_types.append(world.body_type[0])
        star_profile += '-'.join(world_types); profile_parts.append(star_profile)
    return ':'.join(profile_parts)

def generate_atmosphere(size: int) -> int:
    return 0 if size == 0 else max(0, Utils.D6(2) - 7 + size)

def generate_hydrographics(size: int, atmosphere: int) -> int:
    if size <= 1: return 0
    hydro = Utils.D6(2) - 7 + atmosphere
    if atmosphere <= 1 or atmosphere >= 10: hydro -= 4
    return max(0, min(10, hydro))

def generate_population(dm: int = 0) -> int: return max(0, Utils.D6(2) - 2 + dm)
def generate_government(population: int) -> int: return 0 if population == 0 else max(0, Utils.D6(2) - 7 + population)
def generate_law_level(government: int) -> int: return 0 if government == 0 else max(0, Utils.D6(2) - 7 + government)
def generate_starport(population: int) -> str:
    roll = Utils.D6(2) + (2 if population >= 10 else (1 if population >= 8 else (-1 if population <= 4 else (-2 if population <= 2 else 0))))
    return 'X' if roll <= 2 else ('E' if roll <= 4 else ('D' if roll <= 6 else ('C' if roll <= 8 else ('B' if roll <= 10 else 'A'))))
def generate_tech_level(starport: str, size: int, atmosphere: int, hydrographics: int, population: int) -> int:
    dm = {'A': 6, 'B': 4, 'C': 2, 'X': -4}.get(starport, 0) + (2 if size <= 1 else (1 if size <= 4 else 0)) + (1 if atmosphere <= 3 or atmosphere >= 10 else 0) + ({0: 1, 9: 1, 10: 2}.get(hydrographics, 0)) + (4 if population == 10 else (2 if population == 9 else (1 if (1 <= population <= 5 or population == 8) else 0)))
    return max(0, Utils.D6(1) + dm)
def generate_mainworld_uwp(population_dm: int = 0) -> UWP:
    uwp, size_roll = UWP(), Utils.D6(2) - 2
    uwp.size, atm_val = Utils.eHex(size_roll), generate_atmosphere(size_roll)
    uwp.atmosphere, hydro_val = Utils.eHex(atm_val), generate_hydrographics(size_roll, atm_val)
    uwp.hydrographics, pop_val = Utils.eHex(hydro_val), generate_population(population_dm)
    uwp.population, gov_val = Utils.eHex(pop_val), generate_government(pop_val)
    uwp.government, law_val = Utils.eHex(gov_val), generate_law_level(gov_val)
    uwp.law_level, star_val = Utils.eHex(law_val), generate_starport(pop_val)
    uwp.starport = star_val; uwp.tech_level = Utils.eHex(generate_tech_level(star_val, size_roll, atm_val, hydro_val, pop_val))
    return uwp
def calculate_mean_temperature(world: PlanetaryBody, system: StellarSystem) -> float:
    albedo = 0.3 + (0.1 if (Utils.from_eHex(world.hydrographics_code) if world.hydrographics_code.isdigit() else 5) < 2 else (-0.1 if (Utils.from_eHex(world.hydrographics_code) if world.hydrographics_code.isdigit() else 5) > 5 else 0))
    greenhouse = 0.1 * (Utils.from_eHex(world.atmosphere_code) if world.atmosphere_code.isdigit() and 4 <= Utils.from_eHex(world.atmosphere_code) <= 9 else 0)
    parent = next((s for s in system.stars if s.designation == world.parent_star_group), None)
    return 279 * (parent.luminosity * (1 - albedo) * (1 + greenhouse) / world.orbit_au**2)**0.25 if parent and world.orbit_au > 0 else 0.0

def generate_full_system(name="Random System") -> StellarSystem:
    system = StellarSystem(name=name)
    generate_stellar_system_stars(system)
    determine_world_counts(system); calculate_available_orbits(system, model='physics'); calculate_baseline_and_spread(system); handle_anomalies_and_empties(system)
    place_worlds(system, generate_orbital_slots(system)); detail_placed_worlds(system)
    for world in system.all_worlds:
        if world.body_type == 'Terrestrial':
            size_val = Utils.from_eHex(world.size_code)
            atm_val = Utils.from_eHex(world.atmosphere_code) if world.atmosphere_code else generate_atmosphere(size_val)
            world.atmosphere_code = Utils.eHex(atm_val)
            hydro_val = Utils.from_eHex(world.hydrographics_code) if world.hydrographics_code else generate_hydrographics(size_val, atm_val)
            world.hydrographics_code = Utils.eHex(hydro_val); world.mean_temperature = calculate_mean_temperature(world, system)
    assign_final_designations(system); flag_points_of_interest(system)
    return system

def create_system_dataframe(system: StellarSystem) -> pd.DataFrame:
    records = []
    for world in system.all_worlds:
        records.append({'Primary': world.parent_star_group, 'Object': world.designation, 'Name': world.name, 'Orbit#': f"{world.orbit_num:.2f}", 'AU': f"{world.orbit_au:.3f}", 'Period': f"{world.period_years:.3f}y", 'SAH/UWP': world.size_code, 'Sub': len(world.satellites), 'Notes': ", ".join(world.notes)})
        for sat in world.satellites: records.append({'Primary': world.designation, 'Object': sat.designation, 'Name': '', 'Orbit#': '-', 'AU': '-', 'Period': '-', 'SAH/UWP': sat.size_code, 'Sub': 0, 'Notes': ", ".join(sat.notes)})
    return pd.DataFrame(records)

## VII. Main Execution Block

In [ ]:
system = generate_full_system("Zed")
print(f"SYSTEM REPORT: {system.name}")
print("-" * 40)
print(f"System Age: {system.age_gyr} Gyr")
print("Stellar Composition:")
for star in system.stars: 
    if not star.is_composite: print(f"  - Star {star.designation}: {star.spectral_type}, Mass: {star.mass:.3f}, Lum: {star.luminosity:.3f}")
print("-" * 40)
print(f"Short Profile: {generate_short_profile(system)}")
print(f"Long Profile: {generate_long_profile(system)}")
print("-" * 40)
display(create_system_dataframe(system))